In [ ]:
import json, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor

try:
    import xgboost as xgb
except ImportError:
    xgb = None

SEED = 42
np.random.seed(SEED)

EPS = 1e-9

# ---- Speed/quality knobs ----
FAST_MODE = True     # True: target <10-20 min
QUICK_MODE = False   # True: very fast but worse MAE
USE_GPU = False      # keep CPU stable unless you know CUDA is ok
EARLY_STOPPING_ROUNDS = 30 if FAST_MODE else 60

# ---- Weights: "sqrt" is a good compromise for MAE ----
WEIGHT_MODE = "sqrt"   # one of: none, raw, sqrt, clip10

# ---- Optional MAE improvements ----
XGB_OOF_SEEDS = [SEED] if QUICK_MODE else [SEED, SEED + 1337]  # bagging seeds
ENABLE_TOP2_ENSEMBLE = True


In [ ]:
def load_data():
    df = pd.read_csv("c13k_selections.csv")
    with open("c13k_problems.json", "r") as f:
        problems_dict = json.load(f)
    return df, problems_dict

df, problems_dict = load_data()
print("df:", df.shape, "problems:", len(problems_dict))
df.head()


In [ ]:
def safe_div(a, b):
    return float(a / (b + EPS))

def norm_probs(p):
    p = np.asarray(p, dtype=float)
    s = float(np.sum(p))
    if s <= 0:
        return p
    return p / (s + EPS)

def clip01(a):
    return np.clip(np.asarray(a, dtype=float), 0.0, 1.0)

def metrics(y_true, y_pred):
    y_pred = clip01(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


In [ ]:
def get_gamble_stats(outcomes):
    probs = norm_probs([o[0] for o in outcomes])
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    ev = float(np.sum(probs * pays))
    var = float(np.sum(probs * (pays - ev) ** 2))
    sd = float(np.sqrt(max(var, 0.0)))

    mn = float(np.min(pays))
    mx = float(np.max(pays))
    rng = float(mx - mn)

    p_gain = float(np.sum(probs[pays > 0])) if np.any(pays > 0) else 0.0
    p_loss = float(np.sum(probs[pays < 0])) if np.any(pays < 0) else 0.0
    p_zero = float(np.sum(probs[pays == 0])) if np.any(pays == 0) else 0.0

    ev_gain = float(np.sum(probs[pays > 0] * pays[pays > 0])) if np.any(pays > 0) else 0.0
    ev_loss = float(np.sum(probs[pays < 0] * pays[pays < 0])) if np.any(pays < 0) else 0.0

    mean_gain = safe_div(ev_gain, p_gain) if p_gain > 0 else 0.0
    mean_loss = safe_div(ev_loss, p_loss) if p_loss > 0 else 0.0

    abs_pays = np.abs(pays)
    mean_abs = float(np.sum(probs * abs_pays))

    max_gain = float(np.max(pays[pays > 0])) if np.any(pays > 0) else 0.0
    max_loss = float(np.min(pays[pays < 0])) if np.any(pays < 0) else 0.0

    if sd > 0:
        z = (pays - ev) / (sd + EPS)
        skew = float(np.sum(probs * z**3))
        kurt = float(np.sum(probs * z**4))
    else:
        skew = 0.0
        kurt = 0.0

    pclip = np.clip(probs, EPS, 1.0)
    ent = float(-np.sum(pclip * np.log(pclip)))

    return {
        "EV": ev, "SD": sd, "Var": var,
        "Min": mn, "Max": mx, "Range": rng,
        "P_Gain": p_gain, "P_Loss": p_loss, "P_Zero": p_zero,
        "EV_Gain": ev_gain, "EV_Loss": ev_loss,
        "Mean_Gain": mean_gain, "Mean_Loss": mean_loss,
        "Mean_Abs": mean_abs,
        "Max_Gain": max_gain, "Max_Loss": max_loss,
        "Skew": skew, "Kurt": kurt,
        "Entropy": ent,
        "Num_Out": float(len(outcomes)),
    }

def psych_ev(outcomes, alpha=0.88, gamma=0.65):
    probs = norm_probs([o[0] for o in outcomes])
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    subj_pays = np.sign(pays) * (np.abs(pays) ** alpha)
    probs_clipped = np.clip(probs, EPS, 1.0)
    weighted_probs = np.exp(-(-np.log(probs_clipped)) ** gamma)

    return float(np.sum(weighted_probs * subj_pays))

def prob_B_better_than_A(outA, outB):
    pA = norm_probs([o[0] for o in outA])
    xA = np.array([o[1] for o in outA], dtype=float)
    pB = norm_probs([o[0] for o in outB])
    xB = np.array([o[1] for o in outB], dtype=float)

    mat = (xB[:, None] > xA[None, :]).astype(float)
    return float(np.sum((pB[:, None] * pA[None, :]) * mat))


In [ ]:
def build_dataset(df, problems_dict):
    raw_cols = ["Ha","La","pHa","Hb","Lb","pHb"]
    have_raw = all(c in df.columns for c in raw_cols)
    have_std = "bRate_std" in df.columns
    has_n = "n" in df.columns

    stat_keys = [
        "EV","SD","Var","Min","Max","Range",
        "P_Gain","P_Loss","P_Zero",
        "EV_Gain","EV_Loss","Mean_Gain","Mean_Loss",
        "Mean_Abs","Max_Gain","Max_Loss",
        "Skew","Kurt","Entropy","Num_Out"
    ]

    feats, y, w, groups = [], [], [], []
    cache = {}

    for row in df.itertuples(index=False):
        pid = str(row.Problem)
        base = cache.get(pid)

        if base is None:
            prob = problems_dict.get(pid)
            if prob is None:
                continue

            outA = prob["A"]
            outB = prob["B"]

            sA = get_gamble_stats(outA)
            sB = get_gamble_stats(outB)

            peA = psych_ev(outA, alpha=0.88, gamma=0.65)
            peB = psych_ev(outB, alpha=0.88, gamma=0.65)
            peA2 = psych_ev(outA, alpha=0.70, gamma=0.90)
            peB2 = psych_ev(outB, alpha=0.70, gamma=0.90)

            base = {}
            for k in stat_keys:
                base[f"{k}_A"] = float(sA[k])
                base[f"{k}_B"] = float(sB[k])
                base[f"{k}_Diff"] = float(sB[k] - sA[k])

            base["Psych_EV_A"] = float(peA)
            base["Psych_EV_B"] = float(peB)
            base["Psych_EV_Diff"] = float(peB - peA)

            base["Psych_EV2_A"] = float(peA2)
            base["Psych_EV2_B"] = float(peB2)
            base["Psych_EV2_Diff"] = float(peB2 - peA2)

            # safer CV + ratios (avoid division explosions around 0)
            base["CV_A"] = float(sA["SD"] / (abs(sA["EV"]) + 1.0))
            base["CV_B"] = float(sB["SD"] / (abs(sB["EV"]) + 1.0))
            base["CV_Diff"] = float(base["CV_B"] - base["CV_A"])

            base["EV_RatioSoft"] = float(sB["EV"] / (abs(sA["EV"]) + 1.0))
            base["SD_RatioSoft"] = float(sB["SD"] / (abs(sA["SD"]) + 1.0))
            base["Range_RatioSoft"] = float(sB["Range"] / (abs(sA["Range"]) + 1.0))
            base["Entropy_RatioSoft"] = float(sB["Entropy"] / (abs(sA["Entropy"]) + 1.0))

            base["P_B_better_A"] = float(prob_B_better_than_A(outA, outB))

            cache[pid] = base

        rec = dict(base)

        # design vars
        rec["Amb"] = int(bool(row.Amb))
        rec["Corr"] = int(row.Corr)
        rec["Feedback"] = int(bool(row.Feedback))
        rec["LotShapeB"] = int(row.LotShapeB)
        rec["LotNumB"] = int(row.LotNumB)
        rec["Block"] = int(row.Block)

        if have_raw:
            rec["Ha"]  = float(row.Ha); rec["La"]  = float(row.La); rec["pHa"] = float(row.pHa)
            rec["Hb"]  = float(row.Hb); rec["Lb"]  = float(row.Lb); rec["pHb"] = float(row.pHb)
            rec["EV_A_raw"] = float(row.pHa * row.Ha + (1.0 - row.pHa) * row.La)
            rec["EV_B_raw"] = float(row.pHb * row.Hb + (1.0 - row.pHb) * row.Lb)
            rec["EV_raw_Diff"] = float(rec["EV_B_raw"] - rec["EV_A_raw"])

        # interactions
        rec["Amb_x_SD_Diff"] = rec["Amb"] * rec["SD_Diff"]
        rec["Feedback_x_EV_Diff"] = rec["Feedback"] * rec["EV_Diff"]
        rec["Amb_x_Range_Diff"] = rec["Amb"] * rec["Range_Diff"]
        rec["Feedback_x_PGain_Diff"] = rec["Feedback"] * rec["P_Gain_Diff"]

        # abs diffs
        rec["EV_Diff_Abs"] = abs(rec["EV_Diff"])
        rec["SD_Diff_Abs"] = abs(rec["SD_Diff"])
        rec["Entropy_Diff_Abs"] = abs(rec["Entropy_Diff"])
        rec["Psych_EV_Diff_Abs"] = abs(rec["Psych_EV_Diff"])

        # target
        y.append(float(row.bRate))
        groups.append(pid)

        # weights
        n = float(row.n) if has_n else 1.0
        if have_std:
            std = float(row.bRate_std)
            # base reliability weight (not too extreme)
            noise_factor = 1.0 / (std*std + 1e-3)
            noise_factor = min(noise_factor, 25.0)
            w.append(n * noise_factor)
        else:
            w.append(n)

        feats.append(rec)

    X = pd.DataFrame(feats).replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    y = np.array(y, dtype=float)
    w = np.array(w, dtype=float)
    groups = np.array(groups)
    return X, y, w, groups

X, y, w, groups = build_dataset(df, problems_dict)
print(f"Samples: {len(X)} | Features: {X.shape[1]}")


In [ ]:
if WEIGHT_MODE == "none":
    w_model = np.ones_like(w, dtype=float)
elif WEIGHT_MODE == "raw":
    w_model = w.astype(float)
elif WEIGHT_MODE == "sqrt":
    w_model = np.sqrt(np.clip(w, EPS, None))
elif WEIGHT_MODE == "clip10":
    w_model = np.clip(w, 1.0, 10.0)
else:
    raise ValueError("WEIGHT_MODE must be one of: none/raw/sqrt/clip10")

print(
    f"Weight mode: {WEIGHT_MODE} | "
    f"min={w_model.min():.3f}, median={np.median(w_model):.3f}, max={w_model.max():.3f}"
)


In [ ]:
if QUICK_MODE:
    TUNE_SPLITS, FINAL_SPLITS = 2, 3
else:
    TUNE_SPLITS, FINAL_SPLITS = (3, 5)

tune_splits  = list(GroupKFold(n_splits=TUNE_SPLITS ).split(np.zeros(len(y)), y, groups=groups))
final_splits = list(GroupKFold(n_splits=FINAL_SPLITS).split(np.zeros(len(y)), y, groups=groups))
print("Tune folds:", len(tune_splits), "Final folds:", len(final_splits))


In [ ]:
def _sample_max_features(rng):
    # prevents numpy casting floats to strings
    if rng.rand() < 0.5:
        return str(rng.choice(["sqrt", "log2"]))
    else:
        return float(rng.choice([0.6, 0.8, 1.0]))

def sample_rf_params(rng):
    if QUICK_MODE:
        n_estimators = int(rng.choice([300, 500]))
        max_depth = rng.choice([None, 12, 16])
    elif FAST_MODE:
        n_estimators = int(rng.choice([400, 700, 1000]))
        max_depth = rng.choice([None, 10, 14, 18, 24])
    else:
        n_estimators = int(rng.choice([700, 1000, 1400]))
        max_depth = rng.choice([None, 12, 16, 20, 26])

    return {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": int(rng.choice([1, 2, 3, 4])),
        "min_samples_split": int(rng.choice([2, 5, 10])),
        "max_features": _sample_max_features(rng),
        "bootstrap": bool(rng.choice([True, False])),
    }

def build_rf(params):
    return RandomForestRegressor(random_state=SEED, n_jobs=1, **params)

def sample_et_params(rng):
    if QUICK_MODE:
        n_estimators = int(rng.choice([300, 500]))
        max_depth = rng.choice([None, 12, 16])
    elif FAST_MODE:
        n_estimators = int(rng.choice([400, 700, 1000]))
        max_depth = rng.choice([None, 10, 14, 18, 24])
    else:
        n_estimators = int(rng.choice([700, 1000, 1400]))
        max_depth = rng.choice([None, 12, 16, 20, 26])

    return {
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": int(rng.choice([1, 2, 3, 4])),
        "min_samples_split": int(rng.choice([2, 5, 10])),
        "max_features": _sample_max_features(rng),
    }

def build_et(params):
    return ExtraTreesRegressor(random_state=SEED, n_jobs=1, **params)

def sample_mlp_params(rng):
    if QUICK_MODE:
        hidden_options = [(64, 32), (128, 64)]
        max_iters = [180, 250]
    elif FAST_MODE:
        hidden_options = [(64, 32), (128, 64), (128, 64, 32), (256, 128)]
        max_iters = [250, 350]
    else:
        hidden_options = [(64, 32), (128, 64), (128, 64, 32), (256, 128), (256, 128, 64)]
        max_iters = [350, 500, 650]
    return {
        "hidden_layer_sizes": hidden_options[int(rng.randint(len(hidden_options)))],
        "alpha": float(rng.choice([1e-6, 1e-5, 1e-4, 1e-3])),
        "learning_rate_init": float(rng.choice([1e-4, 3e-4, 5e-4, 1e-3, 2e-3])),
        "batch_size": int(rng.choice([32, 64, 128])),
        "max_iter": int(rng.choice(max_iters)),
    }

def build_mlp(params):
    return MLPRegressor(
        random_state=SEED,
        activation="relu",
        solver="adam",
        early_stopping=True,
        n_iter_no_change=12,
        validation_fraction=0.1,
        **params
    )

def sample_xgb_params(rng):
    # πιο "MAE-friendly" και λιγότερο variance από max_depth=7/8
    if QUICK_MODE:
        n_est = [900, 1200]
    elif FAST_MODE:
        n_est = [1200, 1600, 2000]
    else:
        n_est = [1600, 2200, 3000]

    return {
        "n_estimators": int(rng.choice(n_est)),
        "max_depth": int(rng.choice([4, 5, 6])),
        "learning_rate": float(rng.choice([0.01, 0.015, 0.02])),
        "subsample": float(rng.choice([0.75, 0.85, 1.0])),
        "colsample_bytree": float(rng.choice([0.75, 0.85, 1.0])),
        "min_child_weight": float(rng.choice([1.0, 2.0, 4.0, 8.0])),
        "reg_lambda": float(rng.choice([1.0, 2.0, 5.0])),
        "reg_alpha": float(rng.choice([0.0, 0.05, 0.1, 0.5])),
        "gamma": float(rng.choice([0.0, 0.05, 0.1])),
    }

def build_xgb(params):
    if xgb is None:
        raise ImportError("xgboost not installed. pip install xgboost")

    base = {
        "random_state": SEED,
        "n_jobs": -1,
        "objective": "reg:absoluteerror",  # <-- καλύτερο για MAE αν το υποστηρίζει η έκδοση
        "eval_metric": "mae",
        "tree_method": "hist",
    }
    if USE_GPU:
        base["device"] = "cuda"
    return xgb.XGBRegressor(**base, **params)


In [ ]:
def fit_with_weights(model, X, y, sample_weight, X_val=None, y_val=None, val_weight=None):
    module = model.__class__.__module__

    if module.startswith("xgboost"):
        fit_kwargs = {"sample_weight": sample_weight, "verbose": False}
        if X_val is not None and y_val is not None:
            fit_kwargs["eval_set"] = [(X_val, y_val)]
            fit_kwargs["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
            # some versions accept this, some don't:
            if val_weight is not None:
                try:
                    fit_kwargs["sample_weight_eval_set"] = [val_weight]
                except Exception:
                    pass
        try:
            model.fit(X, y, **fit_kwargs)
            return model
        except TypeError:
            # remove optional args if incompatible
            fit_kwargs.pop("sample_weight_eval_set", None)
            fit_kwargs.pop("early_stopping_rounds", None)
            fit_kwargs.pop("eval_set", None)
            model.fit(X, y, sample_weight=sample_weight, verbose=False)
            return model

    # sklearn models
    try:
        model.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        model.fit(X, y)
    return model


In [ ]:
def cv_mae(build_fn, params, X, y, w, splits, scale=False):
    Xv = X.values
    fold_maes = []

    for tr_i, val_i in splits:
        X_tr = Xv[tr_i]; X_val = Xv[val_i]
        y_tr = y[tr_i];  y_val = y[val_i]
        w_tr = w[tr_i];  w_val = w[val_i]

        scaler = None
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

        model = build_fn(params)
        model = fit_with_weights(model, X_tr, y_tr, w_tr, X_val=X_val, y_val=y_val, val_weight=w_val)
        pred = clip01(model.predict(X_val))
        fold_maes.append(mean_absolute_error(y_val, pred))

    return float(np.mean(fold_maes)), float(np.std(fold_maes, ddof=1))

def cv_oof(build_fn, params, X, y, w, splits, scale=False):
    Xv = X.values
    oof = np.zeros_like(y, dtype=float)

    for tr_i, val_i in splits:
        X_tr = Xv[tr_i]; X_val = Xv[val_i]
        y_tr = y[tr_i];  y_val = y[val_i]
        w_tr = w[tr_i];  w_val = w[val_i]

        scaler = None
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

        model = build_fn(params)
        model = fit_with_weights(model, X_tr, y_tr, w_tr, X_val=X_val, y_val=y_val, val_weight=w_val)
        oof[val_i] = model.predict(X_val)

    return clip01(oof)

def cv_oof_xgb_bagged(params, X, y, w, splits, seeds):
    Xv = X.values
    oof = np.zeros_like(y, dtype=float)

    for tr_i, val_i in splits:
        X_tr = Xv[tr_i]; X_val = Xv[val_i]
        y_tr = y[tr_i];  y_val = y[val_i]
        w_tr = w[tr_i];  w_val = w[val_i]

        preds = np.zeros(len(val_i), dtype=float)
        for s in seeds:
            m = build_xgb(params)
            try:
                m.set_params(random_state=int(s))
            except Exception:
                pass
            m = fit_with_weights(m, X_tr, y_tr, w_tr, X_val=X_val, y_val=y_val, val_weight=w_val)
            preds += m.predict(X_val)

        oof[val_i] = preds / float(len(seeds))

    return clip01(oof)


In [ ]:
# ----------- Budget (για να μην κάνει 40') -----------
if QUICK_MODE:
    TRIALS_RF, TRIALS_ET, TRIALS_MLP = 1, 1, 1
    TRIALS_XGB_SCREEN = 6
    TRIALS_XGB_REFINE = 18
elif FAST_MODE:
    TRIALS_RF, TRIALS_ET, TRIALS_MLP = 2, 2, 1   # μικρά, απλά για baseline
    TRIALS_XGB_SCREEN = 10
    TRIALS_XGB_REFINE = 35                        # εδώ δίνουμε budget
else:
    TRIALS_RF, TRIALS_ET, TRIALS_MLP = 4, 4, 2
    TRIALS_XGB_SCREEN = 18
    TRIALS_XGB_REFINE = 60

# ----------- Screening folds (γρήγορα) -----------
screen_splits = list(GroupKFold(n_splits=3).split(np.zeros(len(y)), y, groups=groups))

# ----------- Final folds (ακριβό, μόνο για winner) -----------
final_splits = list(GroupKFold(n_splits=5).split(np.zeros(len(y)), y, groups=groups))

MODEL_REGISTRY = {
    "RandomForest": {"build": build_rf, "sample": sample_rf_params, "scale": False, "trials": TRIALS_RF},
    "ExtraTrees":   {"build": build_et, "sample": sample_et_params, "scale": False, "trials": TRIALS_ET},
    "MLP":          {"build": build_mlp, "sample": sample_mlp_params, "scale": True,  "trials": TRIALS_MLP},
    "XGBoost":      {"build": build_xgb,"sample": sample_xgb_params,"scale": False, "trials": TRIALS_XGB_SCREEN},
}

def tune_model(cfg, X, y, w, splits):
    rng = np.random.RandomState(SEED)
    best = {"mae": 1e9, "std": None, "params": None}
    for _ in range(cfg["trials"]):
        p = cfg["sample"](rng)
        mae, std = cv_mae(cfg["build"], p, X, y, w, splits, scale=cfg["scale"])
        if mae < best["mae"]:
            best = {"mae": mae, "std": std, "params": p}
    return best

# ===== 1) Screening: RF/ET/MLP/XGB (3-fold μόνο) =====
screen_results = []
best_params = {}
oof_screen = {}

for name, cfg in MODEL_REGISTRY.items():
    best = tune_model(cfg, X, y, w_model, screen_splits)
    best_params[name] = best
    # 3-fold OOF μόνο για screening (γρήγορα)
    if name == "XGBoost":
        oof = cv_oof_xgb_bagged(best["params"], X, y, w_model, screen_splits, seeds=[SEED])
    else:
        oof = cv_oof(cfg["build"], best["params"], X, y, w_model, screen_splits, scale=cfg["scale"])
    oof_screen[name] = oof
    mae, rmse, r2 = metrics(y, oof)
    screen_results.append((name, mae, rmse, r2, best["params"]))

screen_df = pd.DataFrame(screen_results, columns=["Model","MAE_3fold","RMSE_3fold","R2_3fold","Best_Params"]).sort_values("MAE_3fold").reset_index(drop=True)
display(screen_df)

# ===== 2) Refine: ΜΟΝΟ XGBoost με περισσότερα trials (3-fold) =====
print("\nRefining XGBoost only...")
xgb_cfg = {"build": build_xgb, "sample": sample_xgb_params, "scale": False, "trials": TRIALS_XGB_REFINE}

rng = np.random.RandomState(SEED)
best_xgb = {"mae": 1e9, "std": None, "params": None}
for _ in range(TRIALS_XGB_REFINE):
    p = sample_xgb_params(rng)
    mae, std = cv_mae(build_xgb, p, X, y, w_model, screen_splits, scale=False)
    if mae < best_xgb["mae"]:
        best_xgb = {"mae": mae, "std": std, "params": p}

print("Best XGB (screen 3-fold) MAE:", round(best_xgb["mae"], 6))
print("Params:", best_xgb["params"])

# ===== 3) Final: 5-fold OOF ΜΟΝΟ για XGBoost (με 1 seed ή 2 seeds) =====
final_oof_xgb = cv_oof_xgb_bagged(best_xgb["params"], X, y, w_model, final_splits, seeds=XGB_OOF_SEEDS)
final_mae, final_rmse, final_r2 = metrics(y, final_oof_xgb)

print("\nFINAL 5-fold:")
print("XGBoost | MAE =", round(final_mae, 6), "| RMSE =", round(final_rmse, 6), "| R2 =", round(final_r2, 6))


In [ ]:
best_single_model = str(res_df.iloc[0]["Model"])
best_single_mae = float(res_df.iloc[0]["CV_MAE"])

best_mae_final = best_single_mae
best_name_final = best_single_model
best_oof_final = oof_preds[best_single_model]

if ENABLE_TOP2_ENSEMBLE and len(res_df) >= 2:
    a = str(res_df.iloc[0]["Model"])
    b = str(res_df.iloc[1]["Model"])

    best_ens = (1e9, None, None, None)
    for wgt in [0.2,0.3,0.4,0.5,0.6,0.7,0.8]:
        ens = wgt*oof_preds[a] + (1.0-wgt)*oof_preds[b]
        mae, rmse, r2 = metrics(y, ens)
        if mae < best_ens[0]:
            best_ens = (mae, rmse, r2, wgt)

    if best_ens[0] < best_mae_final:
        best_mae_final = float(best_ens[0])
        best_name_final = f"Ensemble({a},{b})"
        best_oof_final = wgt*oof_preds[a] + (1.0-wgt)*oof_preds[b]
        print(f"Ensemble improved MAE: {best_mae_final:.6f} using {a}/{b} w={best_ens[3]:.2f}")
    else:
        print("Ensemble did not improve over best single model.")

print(f"Best final: {best_name_final} | MAE={best_mae_final:.6f}")
print(f"{best_mae_final:.6f}")  # one-line MAE (assignment style)


In [ ]:
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

best_single_model = str(res_df.iloc[0]["Model"])
cfg = MODEL_REGISTRY[best_single_model]
params = best_params[best_single_model]["params"]

# train final on full data
X_fit = X.values
scaler = None
if cfg["scale"]:
    scaler = StandardScaler()
    X_fit = scaler.fit_transform(X_fit)

final_model = cfg["build"](params)
final_model = fit_with_weights(final_model, X_fit, y, w_model)

def plot_feature_importance(feature_names, scores, topk=15, title="Feature importance", save_path=None):
    fi = (pd.DataFrame({"Feature": feature_names, "Score": scores})
          .sort_values("Score", ascending=False).head(topk))
    plt.figure(figsize=(9,5))
    plt.barh(fi["Feature"][::-1], fi["Score"][::-1])
    plt.title(title)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fi

fi = None
fi_save = ARTIFACTS_DIR / f"feature_importance_{best_single_model.lower()}.png"

if hasattr(final_model, "feature_importances_"):
    fi = plot_feature_importance(X.columns.tolist(), final_model.feature_importances_, topk=15,
                                 title=f"{best_single_model} feature importance", save_path=fi_save)
else:
    # permutation importance for models without native importance
    def neg_mae(est, X_in, y_in):
        pred = clip01(est.predict(X_in))
        return -mean_absolute_error(y_in, pred)

    class Wrapper:
        def __init__(self, model, scaler):
            self.model = model
            self.scaler = scaler
        def fit(self, X_in, y_in=None):
            return self
        def predict(self, X_in):
            Xp = self.scaler.transform(X_in) if self.scaler is not None else X_in
            return self.model.predict(Xp)

    # sample to keep it fast
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(X), size=min(1500, len(X)), replace=False)
    Xs = X.iloc[idx].values
    ys = y[idx]

    wrapped = Wrapper(final_model, scaler)
    imp = permutation_importance(wrapped, Xs, ys, scoring=neg_mae, n_repeats=2, random_state=SEED)
    fi = plot_feature_importance(X.columns.tolist(), imp.importances_mean, topk=15,
                                 title=f"Permutation importance ({best_single_model})", save_path=fi_save)

if fi is not None:
    display(fi)
    fi.to_csv(ARTIFACTS_DIR / "feature_importance_top15.csv", index=False)

# save results + oof
res_df.to_csv(ARTIFACTS_DIR / "cv_results.csv", index=False)

# best single OOF
oof_best_single = oof_preds[best_single_model]
pd.DataFrame({"Problem": groups, "y_true": y, "oof_pred": oof_best_single}).to_csv(
    ARTIFACTS_DIR / "oof_predictions_best_single.csv", index=False
)

report = {
    "best_single_model": best_single_model,
    "best_single_mae": float(res_df.iloc[0]["CV_MAE"]),
    "best_final_name": best_name_final,
    "best_final_mae": float(best_mae_final),
    "seed": int(SEED),
    "weight_mode": WEIGHT_MODE,
    "xgb_oof_seeds": XGB_OOF_SEEDS,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "best_params": best_params[best_single_model]["params"],
    "n_samples": int(len(y)),
    "n_features_used": int(X.shape[1]),
}
with open(ARTIFACTS_DIR / "report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

artifact = {
    "model": final_model,
    "scaler": scaler,
    "feature_names": list(X.columns),
    "best_single_model": best_single_model,
    "best_params": best_params[best_single_model]["params"],
    "seed": SEED,
    "weight_mode": WEIGHT_MODE,
}
joblib.dump(artifact, ARTIFACTS_DIR / "best_model.joblib")

print("Artifacts saved in:", ARTIFACTS_DIR.resolve())
